# Water Potability — Baseline Models

A first benchmark before any tuning. The goal isn't a good model — it's an honest number to measure future work against.

**From EDA:** no feature showed meaningful linear correlation with the target (all |r| < 0.05), and the majority class is 61%. Two things to check — whether linear models fail as that predicts, and whether anything meaningfully beats the 61% floor.

**On the metric:** the costly error is unsafe water predicted as safe. But "optimise recall on the unsafe class" turns out to be unsound, and the section below shows why. Four metrics were tried and eliminated before settling on one.

In [1]:
!pip install xgboost --quiet

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
import pickle

In [2]:
path = '/kaggle/input/datasets/adityakadiwal/water-potability/water_potability.csv'
df = pd.read_csv(path)

for col in ['ph', 'Sulfate', 'Trihalomethanes']:
    df[col] = df[col].fillna(df[col].median())

print("Missing after imputation:", df.isnull().sum().sum())

X = df.drop('Potability', axis=1)
y = df['Potability']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Missing after imputation: 0
Train: (2620, 9) Test: (656, 9)


## Preprocessing

Median imputation on `ph`, `Sulfate`, `Trihalomethanes`. EDA showed all three near-symmetric (|skew| < 0.1), so mean would give nearly identical results — median is a safe default, not a requirement of the data.

80/20 split, stratified to preserve the 61/39 class ratio. 2,620 train / 656 test.

In [3]:

models = {
    'Dummy': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(f"{name}: acc={accuracy_score(y_test, pred):.3f} | "
          f"rec_unsafe={recall_score(y_test, pred, pos_label=0):.3f} | "
          f"rec_safe={recall_score(y_test, pred, pos_label=1):.3f} | "
          f"f1_unsafe={f1_score(y_test, pred, pos_label=0):.3f} | "
          f"macro_f1={f1_score(y_test, pred, average='macro'):.3f}")

Dummy: acc=0.610 | rec_unsafe=1.000 | rec_safe=0.000 | f1_unsafe=0.758 | macro_f1=0.379


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression: acc=0.610 | rec_unsafe=1.000 | rec_safe=0.000 | f1_unsafe=0.758 | macro_f1=0.379
RandomForest: acc=0.659 | rec_unsafe=0.887 | rec_safe=0.301 | f1_unsafe=0.760 | macro_f1=0.584
XGBoost: acc=0.642 | rec_unsafe=0.797 | rec_safe=0.398 | f1_unsafe=0.731 | macro_f1=0.598


## Results

| Model | Acc | Rec (unsafe) | Rec (safe) | F1 (unsafe) | Macro F1 |
|---|---|---|---|---|---|
| Dummy | 0.610 | 1.000 | 0.000 | 0.758 | 0.379 |
| LogisticRegression | 0.610 | 1.000 | 0.000 | 0.758 | 0.379 |
| RandomForest | 0.659 | 0.887 | 0.301 | 0.760 | 0.584 |
| XGBoost | 0.642 | 0.797 | 0.398 | 0.731 | **0.598** |

### Choosing a metric

Each candidate was checked against the dummy first. If a model that ignores every feature scores well, the metric isn't measuring what it appears to.

| Metric | Dummy | Best model | Verdict |
|---|---|---|---|
| Accuracy | 0.610 | 0.659 | Ruled out — matched by a constant predictor |
| Recall (potable) | 0.000 | 0.398 | Wrong class — the safety risk is a class 0 miss |
| Recall (not potable) | **1.000** | 0.887 | Ruled out — dummy scores perfectly by labelling everything unsafe |
| F1 (not potable) | 0.758 | 0.760 | Ruled out — a 0.002 gap is noise |
| **Macro F1** | **0.379** | **0.598** | **Selected** |

Macro F1 averages per-class F1 with equal weight, so a model cannot score well by handling only the majority class. The dummy's 0.000 F1 on the potable class drags its macro F1 to 0.379, well clear of both tree models.

### Other findings

**LogisticRegression matched the dummy exactly** — confirming the EDA prediction that with no linear feature-target relationship, a linear model has nothing to work with.

**Convergence warning:** LogisticRegression hit its 1,000-iteration limit. Cause is feature scale — `Solids` reaches ~61,000 while `Turbidity` stays under 7. Tree models are scale-invariant, so only this one complained. Whether `StandardScaler` changes the result or merely silences the warning is untested.

In [4]:
for name in ['RandomForest', 'XGBoost']:
    print(f"--- {name} ---")
    print(classification_report(y_test, models[name].predict(X_test),
                                target_names=['Not Potable', 'Potable']))

--- RandomForest ---
              precision    recall  f1-score   support

 Not Potable       0.66      0.89      0.76       400
     Potable       0.63      0.30      0.41       256

    accuracy                           0.66       656
   macro avg       0.65      0.59      0.58       656
weighted avg       0.65      0.66      0.62       656

--- XGBoost ---
              precision    recall  f1-score   support

 Not Potable       0.67      0.80      0.73       400
     Potable       0.56      0.40      0.46       256

    accuracy                           0.64       656
   macro avg       0.62      0.60      0.60       656
weighted avg       0.63      0.64      0.63       656



## Per-class breakdown

| | Precision | Recall | |
|---|---|---|---|
| RandomForest | 0.66 / 0.63 | 0.89 / 0.30 | (unsafe / safe) |
| XGBoost | 0.67 / 0.56 | 0.80 / 0.40 | (unsafe / safe) |

Precision is similar across both models; recall differs sharply. RandomForest leans toward predicting unsafe, XGBoost splits more evenly. That balance is what macro F1 rewards, and it's the entire source of the 0.014 gap between them.

In [5]:
y_shuffled = y_train.sample(frac=1, random_state=42).reset_index(drop=True)

print("REAL LABELS")
print("  RandomForest — acc: 0.659, macro F1: 0.584")
print("  XGBoost      — acc: 0.642, macro F1: 0.598")
print()
print("SHUFFLED LABELS")

for name, model in [('RandomForest', RandomForestClassifier(random_state=42)),
                    ('XGBoost', XGBClassifier(random_state=42))]:
    model.fit(X_train, y_shuffled)
    pred = model.predict(X_test)
    print(f"  {name} — acc: {accuracy_score(y_test, pred):.3f}, "
          f"macro F1: {f1_score(y_test, pred, average='macro'):.3f}")

REAL LABELS
  RandomForest — acc: 0.659, macro F1: 0.584
  XGBoost      — acc: 0.642, macro F1: 0.598

SHUFFLED LABELS
  RandomForest — acc: 0.585, macro F1: 0.466
  XGBoost — acc: 0.527, macro F1: 0.479


## Shuffle test — is there real signal?

Training labels were randomly shuffled, destroying any feature-label relationship, then the same models were retrained and evaluated on the real test set. Whatever they learn from shuffled labels is noise by construction.

| | Real macro F1 | Shuffled macro F1 | Gap |
|---|---|---|---|
| RandomForest | 0.584 | 0.466 | **+0.118** |
| XGBoost | 0.598 | 0.479 | **+0.119** |

Both models drop ~0.12 when the relationship is destroyed, consistently. **Real signal exists.** Had the gap been near zero, the earlier results would have been meaningless.

The shuffled models still score ~0.47 rather than 0 — a model trained on garbage still predicts something, and on a 61/39 split that lands non-trivially. 0.47 is the true "learned nothing" floor, not 0.

The signal is real but modest, which sets expectations for tuning.

In [6]:
best = models['XGBoost']
with open('model.pkl', 'wb') as f:
    pickle.dump(best, f)
print("saved")

saved
